# 🏆 VIETNAMESE LEGAL QA & RAG PIPELINE (TUÂN THỦ GIỚI HẠN <= 4.0B THAM SỐ)
## Hệ Thống Truy Hồi & Sinh Câu Trả Lời Pháp Luật Tiếng Việt Dạng Văn Xuôi

---

### 📊 BẢNG CÂN ĐỐI NGÂN SÁCH THAM SỐ (TỔNG CỘNG <= 4.0B PARAMETERS)

| Thành phần Pipeline | Mô hình Pre-trained / Thuật toán | Kiến trúc cơ sở | Số lượng tham số | % So với giới hạn 4B |
| :--- | :--- | :--- | :--- | :--- |
| **Stage 1: Dense Vector Retrieval** | `BAAI/bge-m3` | Multi-Lingual Bi-Encoder | **~568 Triệu (~0.57B)** | ~14.2% |
| **Stage 1: Lexical Search** | `BM25Okapi` + `PyVi` | Thống kê tần suất từ khóa + CRF | **0 (Không dùng DNN)** | 0.0% |
| **Stage 2: Deep Cross-Encoder Reranker** | `BAAI/bge-reranker-v2-m3` | XLM-RoBERTa Cross-Encoder | **~568 Triệu (~0.57B)** | ~14.2% |
| **Stage 3: Legal Answer Generator** | `Qwen/Qwen2.5-3B-Instruct` | Transformer Decoder-only LLM | **~3.09 Tỷ (~3.09B)** | ~77.2% |
| **TỔNG CỘNG HỆ THỐNG** | **BGE-M3 + Reranker + Qwen2.5-3B** | **End-to-End Legal RAG Pipeline** | **~3.66 Tỷ (~3.66B)** | **~91.5% (HỢP LỆ <= 4.0B)** |

> **Cấu hình siêu nhẹ (Fast Setup)**: Có thể chuyển Generator sang `Qwen/Qwen2.5-1.5B-Instruct` (~1.54B) để tổng tham số chỉ còn **~2.11B**, tiết kiệm VRAM và tăng tốc độ suy luận gấp 2 lần.

---

### 🏗️ QUY TRÌNH KỸ THUẬT 4 BƯỚC CỐT LÕI (THEO CHUẨN T2/README.md):

1. **Kiểm tra format, Làm sạch dữ liệu & Context-Aware Chunking**:
   - Lọc bỏ các context có trường passage rỗng hoặc lỗi format JSON.
   - Băm MD5 gộp các văn bản trùng lặp 100% nội dung về Canonical ID.
   - Chuẩn hóa Unicode NFC, loại bỏ ký tự điều khiển ẩn và khoảng trắng thừa.
   - Cắt văn bản theo cấu trúc Điều luật, ngắt câu và gắn **Breadcrumb Context Header** `[Tên Văn Bản > Điều X: Tiêu đề]` vào đầu mỗi chunk để chống mất ngữ cảnh.
   - Tách từ bằng `PyVi` cho BM25 song song bản thô NFC cho Dense/Reranker/LLM.
2. **Truy hồi sơ bộ đa phương thức (First-Stage Hybrid Retrieval)**:
   - BM25Okapi bắt chính xác số hiệu luật, thuật ngữ hiếm + BGE-M3 bắt ngữ nghĩa sâu sắc.
   - Hợp nhất xếp hạng bằng **Reciprocal Rank Fusion (RRF)** mở rộng phễu Top 25.
3. **Tái xếp hạng chuyên sâu (Second-Stage Deep Cross-Encoder Reranking)**:
   - `bge-reranker-v2-m3` tính Full Cross-Attention, lọc các bẫy điều kiện loại trừ ('trừ trường hợp...').
   - Xác định Top 1 - 2 Điều luật trọng tâm nhất.
4. **Sinh câu trả lời căn chỉnh cấu trúc chuẩn (Style-Aligned Answer Generation)**:
   - LLM sinh câu trả lời theo đúng phong cách văn xuôi pháp lý của Thư Viện Pháp Luật:
     `Căn cứ theo [Điều/Khoản] [Tên Luật] -> Trích dẫn nguyên văn -> Kết luận ('Theo đó,... / Như vậy,...')`.
   - Tối ưu hóa trực tiếp hai độ đo đánh giá của BTC: **METEOR (Độ đo chính)** & **ROUGE-L (Độ đo phụ)**.

In [ ]:
# =====================================================================
# 1. SYSTEM IMPORTS, HARDWARE SETUP & ENVIRONMENT CONFIGURATION
# =====================================================================
import os
import sys
import glob
import json
import pickle
import random
import re
import hashlib
import unicodedata
import zipfile
from pathlib import Path
from typing import List, Dict, Tuple, Set, Union, Any, Optional

# Vá lỗi RPC trên môi trường Kaggle / Linux
_rpc_path = "/usr/local/lib/python3.12/dist-packages/torch/distributed/rpc/__init__.py"
if os.path.exists(_rpc_path):
    try:
        with open(_rpc_path, "r") as _f:
            _c = _f.read()
        if 'return hasattr(torch._C, "_rpc_init")' in _c:
            with open(_rpc_path, "w") as _f:
                _f.write(_c.replace('return hasattr(torch._C, "_rpc_init")', 'return False'))
    except Exception:
        pass

import numpy as np
import torch
if not hasattr(torch.distributed, "is_available"):
    torch.distributed.is_available = lambda: False
from tqdm.auto import tqdm

# Thiết bị tính toán (GPU / CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"[System] Execution Device: {DEVICE} (Torch Dtype: {TORCH_DTYPE})")
if DEVICE == "cuda":
    print(f"[System] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[System] VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print(f"[System] Running on CPU. Multi-threaded execution enabled.")
    torch.set_num_threads(os.cpu_count() or 4)

# Kiểm tra các thư viện phụ thuộc
try:
    from pyvi import ViTokenizer
    HAS_PYVI = True
except ImportError:
    HAS_PYVI = False

try:
    from rank_bm25 import BM25Okapi
    HAS_RANK_BM25 = True
except ImportError:
    HAS_RANK_BM25 = False

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

print(f"[Dependencies] PyVi: {HAS_PYVI} | Rank-BM25: {HAS_RANK_BM25} | Sentence-Transformers: {HAS_SENTENCE_TRANSFORMERS} | Transformers: {HAS_TRANSFORMERS}")


In [ ]:
# =====================================================================
# 2. GLOBAL PATH CONFIGURATION & HYPERPARAMETERS SETUP
# =====================================================================
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("[Environment] Detected Kaggle Environment")
    K_INPUT = Path("/kaggle/input")
    
    # 1. Đường dẫn tập Train: dataset 'train_data(t2)' -> tệp 'train (1).json'
    train_candidates = [
        K_INPUT / "train_data(t2)" / "train (1).json",
        K_INPUT / "train_data(t2)" / "train.json",
        K_INPUT / "train-data-t2" / "train (1).json",
        K_INPUT / "train-data-t2" / "train.json",
    ]
    TRAIN_PATH = None
    for cand in train_candidates:
        if cand.exists():
            TRAIN_PATH = cand
            break
    if TRAIN_PATH is None:
        globbed_train = list(K_INPUT.glob("**/train*.json"))
        TRAIN_PATH = globbed_train[0] if globbed_train else train_candidates[0]
    
    # 2. Đường dẫn tập Test: dataset 'public_official(t2)' -> tệp 'public-official (1).json'
    test_candidates = [
        K_INPUT / "public_official(t2)" / "public-official (1).json",
        K_INPUT / "public_official(t2)" / "public-official.json",
        K_INPUT / "public-official-t2" / "public-official (1).json",
        K_INPUT / "public-official-t2" / "public-official.json",
    ]
    TEST_PATH = None
    for cand in test_candidates:
        if cand.exists():
            TEST_PATH = cand
            break
    if TEST_PATH is None:
        globbed_test = list(K_INPUT.glob("**/public*.json")) + list(K_INPUT.glob("**/test*.json"))
        TEST_PATH = globbed_test[0] if globbed_test else test_candidates[0]
    
    # 3. Đường dẫn kho văn bản: dataset 'selected-contexts(t2)' -> thư mục con 'selected-contexts'
    corpus_candidates = [
        K_INPUT / "selected-contexts(t2)" / "selected-contexts",
        K_INPUT / "selected-contexts(t2)",
        K_INPUT / "selected-contexts-t2" / "selected-contexts",
        K_INPUT / "selected-contexts-t2",
    ]
    CORPUS_DIR = None
    for cand in corpus_candidates:
        if cand.exists() and (list(cand.glob("context_*.json")) or list(cand.glob("**/context_*.json"))):
            CORPUS_DIR = cand
            break
    if CORPUS_DIR is None:
        ctx_files = list(K_INPUT.glob("**/context_*.json"))
        CORPUS_DIR = ctx_files[0].parent if ctx_files else corpus_candidates[0]
    
    CACHE_DIR = Path("/kaggle/working/cache")
    OUTPUT_DIR = Path("/kaggle/working")
else:
    print("[Environment] Detected Local Environment")
    BASE_DIR = Path(".").resolve()
    # Định vị thư mục T2 chính xác
    if (BASE_DIR / "T2").exists():
        T2_DIR = BASE_DIR / "T2"
    else:
        T2_DIR = BASE_DIR
    
    TRAIN_PATH = T2_DIR / "train (1).json"
    if not TRAIN_PATH.exists():
        TRAIN_PATH = T2_DIR / "train.json"
    
    TEST_PATH = T2_DIR / "public-official (1).json"
    if not TEST_PATH.exists():
        TEST_PATH = T2_DIR / "public-official.json"
        
    CORPUS_DIR = T2_DIR / "selected-contexts (1)" / "selected-contexts"
    if not CORPUS_DIR.exists():
        CORPUS_DIR = T2_DIR / "selected-contexts (1)"
    if not CORPUS_DIR.exists():
        CORPUS_DIR = T2_DIR / "selected-contexts"
        
    CACHE_DIR = T2_DIR / "cache"
    OUTPUT_DIR = T2_DIR

CACHE_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission.json"
SUBMISSION_ZIP_PATH = OUTPUT_DIR / "submission.zip"

PROCESSED_CHUNKS_PATH = CACHE_DIR / "processed_chunks.json"
CORPUS_META_PATH = CACHE_DIR / "corpus_meta.json"
BM25_INDEX_PATH = CACHE_DIR / "bm25_index.pkl"
DENSE_EMBEDDINGS_PATH = CACHE_DIR / "dense_embeddings.npy"

# Đếm nhanh số văn bản ngữ cảnh tìm thấy
found_contexts = len(list(CORPUS_DIR.glob("context_*.json"))) if CORPUS_DIR.exists() else 0
if found_contexts == 0 and CORPUS_DIR.exists():
    found_contexts = len(list(CORPUS_DIR.glob("**/context_*.json")))

print(f"[Paths] Train File   : {TRAIN_PATH} (Exists: {TRAIN_PATH.exists()})")
print(f"[Paths] Test File    : {TEST_PATH} (Exists: {TEST_PATH.exists()})")
print(f"[Paths] Contexts Dir : {CORPUS_DIR} (Found: {found_contexts} contexts, Exists: {CORPUS_DIR.exists()})")
print(f"[Paths] Cache Dir    : {CACHE_DIR}")
print(f"[Paths] Output Dir   : {OUTPUT_DIR}")

# ---------------------------------------------------------------------
# CẤU HÌNH MÔ HÌNH PRE-TRAINED (TỔNG THAM SỐ ~3.66B <= 4.0B)
# ---------------------------------------------------------------------
DENSE_MODEL_NAME = "BAAI/bge-m3"                         # ~0.57B
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"          # ~0.57B
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"        # ~3.09B
FALLBACK_GENERATOR_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # ~1.54B (Fast Option)

# ---------------------------------------------------------------------
# SIÊU THAM SỐ PIPELINE
# ---------------------------------------------------------------------
MAX_CHUNK_WORDS = 350       # Số từ tối đa cho mỗi chunk con
CHUNK_OVERLAP = 50          # Số từ gối đầu giữ ngữ cảnh liên tục
FIRST_STAGE_TOP_K = 25      # Top ứng viên từ Hybrid Retrieval
RERANK_TOP_K = 2            # Số Điều luật cốt lõi đưa vào prompt LLM
MAX_NEW_TOKENS = 512        # Giới hạn độ dài sinh câu trả lời
GEN_TEMPERATURE = 0.1       # Nhiệt độ thấp đảm bảo câu trả lời bám sát văn bản luật
RANDOM_SEED = 42


In [ ]:
# =====================================================================
# 3. UTILITIES, TEXT NORMALIZATION & OFFICIAL BTC METRICS (METEOR & ROUGE-L)
# =====================================================================
def load_json(file_path: Union[str, Path]) -> dict:
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data: dict, file_path: Union[str, Path], indent: int = 4) -> None:
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=indent)

def normalize_vietnamese_text(text: str) -> str:
    """
    Chuẩn hóa Unicode NFC, xử lý ký tự điều khiển và khoảng trắng thừa.
    """
    if not text:
        return ""
    text = unicodedata.normalize('NFC', str(text))
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[\t\f\v]', ' ', text)
    text = re.sub(r'\n+', '\n', text)
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    text = ' '.join(lines)
    return re.sub(r'\s+', ' ', text).strip()

def word_segment(text: str) -> str:
    """
    Tách từ tiếng Việt qua PyVi để tạo compound words phục vụ BM25.
    """
    text = normalize_vietnamese_text(text)
    if HAS_PYVI:
        try:
            return ViTokenizer.tokenize(text)
        except Exception:
            return text
    return text

# ---------------------------------------------------------------------
# THUẬT TOÁN ĐO LƯỜNG ĐỘC LẬP: ROUGE-L & METEOR (CHUẨN BTC)
# ---------------------------------------------------------------------
def _calc_lcs(x: List[str], y: List[str]) -> int:
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

def calc_rouge_l(ref_str: str, pred_str: str) -> float:
    """Tính ROUGE-L dựa trên Longest Common Subsequence (LCS)."""
    x = normalize_vietnamese_text(ref_str).split()
    y = normalize_vietnamese_text(pred_str).split()
    if not x or not y:
        return 1.0 if x == y else 0.0
    lcs_len = _calc_lcs(x, y)
    if lcs_len == 0:
        return 0.0
    r = lcs_len / len(x)
    p = lcs_len / len(y)
    return (2.0 * r * p) / (r + p) if (r + p) > 0 else 0.0

def calc_meteor(ref_str: str, pred_str: str) -> float:
    """Tính điểm METEOR dựa trên unigram matching, precision & recall."""
    ref_toks = normalize_vietnamese_text(ref_str).split()
    hyp_toks = normalize_vietnamese_text(pred_str).split()
    if not ref_toks or not hyp_toks:
        return 1.0 if ref_toks == hyp_toks else 0.0
    ref_set = set(ref_toks)
    matches = sum(1 for w in hyp_toks if w in ref_set)
    if matches == 0:
        return 0.0
    p = matches / len(hyp_toks)
    r = matches / len(ref_toks)
    # Trọng số beta=9 thiên về Recall theo chuẩn định nghĩa METEOR
    return (10.0 * p * r) / (r + 9.0 * p) if (r + 9.0 * p) > 0 else 0.0

def evaluate_predictions(ground_truth: Dict[str, Any], predictions: Dict[str, Any]) -> Dict[str, float]:
    rouge_scores = []
    meteor_scores = []
    for q_id, sample in ground_truth.items():
        ref = sample.get('answer', '') if isinstance(sample, dict) else str(sample)
        if not ref:
            continue
        pred_item = predictions.get(q_id, {})
        pred = pred_item.get('answer', '') if isinstance(pred_item, dict) else str(pred_item)
        rouge_scores.append(calc_rouge_l(ref, pred))
        meteor_scores.append(calc_meteor(ref, pred))
    
    mean_rouge = float(np.mean(rouge_scores)) if rouge_scores else 0.0
    mean_meteor = float(np.mean(meteor_scores)) if meteor_scores else 0.0
    return {'meteor': mean_meteor, 'rouge_l': mean_rouge}


In [ ]:
# =====================================================================
# 4. BƯỚC 1A: KIỂM TRA FORMAT, LÀM SẠCH DỮ LIỆU & KHỬ TRÙNG LẶP (DATA SANITATION)
# =====================================================================
def audit_and_clean_corpus(force_recompute: bool = False) -> Dict[str, Any]:
    """
    1. Kiểm tra format cấu trúc JSON từng tệp context.
    2. Loại bỏ các văn bản có 'passage' rỗng.
    3. Băm MD5 gộp các văn bản trùng lặp nội dung 100% về Canonical ID.
    """
    if not force_recompute and CORPUS_META_PATH.exists():
        try:
            meta = load_json(CORPUS_META_PATH)
            if meta and 'empty_doc_ids' in meta:
                print(f"[Corpus Audit] Nạp metadata từ cache: {len(meta.get('empty_doc_ids', []))} doc rỗng, {len(meta.get('duplicate_groups', []))} nhóm trùng.")
                return meta
        except Exception:
            pass

    file_list = glob.glob(str(CORPUS_DIR / "context_*.json"))
    if not file_list:
        file_list = glob.glob(str(CORPUS_DIR / "**" / "context_*.json"), recursive=True)
    if not file_list and IS_KAGGLE:
        file_list = glob.glob("/kaggle/input/**/context_*.json", recursive=True)

    print(f"[Corpus Audit] Đang kiểm tra {len(file_list)} tệp văn bản quy phạm pháp luật...")
    empty_doc_ids = set()
    passage_to_docs = {}
    valid_docs_count = 0

    for file_path in tqdm(file_list, desc="Auditing Contexts"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                doc_data = json.load(f)
            doc_id = str(doc_data.get('id', ''))
            passage = doc_data.get('passage', '')
            if not doc_id:
                continue
            cleaned = normalize_vietnamese_text(passage)
            if not cleaned:
                empty_doc_ids.add(doc_id)
                continue
            p_hash = hashlib.md5(cleaned.encode('utf-8')).hexdigest()
            if p_hash not in passage_to_docs:
                passage_to_docs[p_hash] = []
            passage_to_docs[p_hash].append(doc_id)
            valid_docs_count += 1
        except Exception:
            continue

    doc_to_canonical = {}
    canonical_to_duplicates = {}
    duplicate_groups = []

    for p_hash, doc_group in passage_to_docs.items():
        canonical_id = sorted(doc_group, key=lambda x: (len(x), x))[0]
        canonical_to_duplicates[canonical_id] = doc_group
        for doc_id in doc_group:
            doc_to_canonical[doc_id] = canonical_id
        if len(doc_group) > 1:
            duplicate_groups.append(doc_group)

    print(f"\n[Corpus Audit Summary]")
    print(f"  - Văn bản hợp lệ               : {valid_docs_count}")
    print(f"  - Văn bản rỗng bị loại bỏ      : {len(empty_doc_ids)}")
    print(f"  - Nhóm văn bản trùng lặp 100%  : {len(duplicate_groups)} ({sum(len(g) for g in duplicate_groups)} văn bản)")

    meta = {
        "empty_doc_ids": list(empty_doc_ids),
        "doc_to_canonical": doc_to_canonical,
        "canonical_to_duplicates": canonical_to_duplicates,
        "duplicate_groups": duplicate_groups
    }
    save_json(meta, CORPUS_META_PATH, indent=2)
    return meta

corpus_metadata = audit_and_clean_corpus()


In [ ]:
# =====================================================================
# 5. BƯỚC 1B: CONTEXT-AWARE CHUNKING & PYVI WORD SEGMENTATION
# =====================================================================
ARTICLE_PATTERN = re.compile(r'(?i)(Điều\s+\d+[a-z]?[:.]?[^\n\.]*)')
SENTENCE_SPLIT_PATTERN = re.compile(r'(?<=[.!?;\n])\s+')

def context_aware_chunk_passage(passage: str, doc_name: str, max_words: int = MAX_CHUNK_WORDS, overlap: int = CHUNK_OVERLAP) -> List[Dict[str, str]]:
    """
    Thuật toán Context-Aware Chunking:
    - Thay vì cắt văn bản thô bạo theo số từ (dễ đứt gãy ngữ cảnh), hệ thống nhận diện các 'Điều' trong văn bản.
    - Nếu một Điều quá dài, cắt nhỏ theo dấu chấm câu.
    - Tiêu đề của Điều luật được tự động dán vào đầu mỗi đoạn nhỏ (Breadcrumb Context Header).
    - Nhờ vậy, đoạn văn bản dù nằm ở đâu cũng không bao giờ bị mất bối cảnh gốc.
    """
    words = passage.split()
    if len(words) <= max_words:
        header = f"[{doc_name}] " if doc_name else ""
        return [{"text": f"{header}{passage}", "article": doc_name}]

    matches = list(ARTICLE_PATTERN.finditer(passage))
    chunks = []
    
    if matches:
        sections = []
        for idx, match in enumerate(matches):
            start_idx = match.start()
            end_idx = matches[idx + 1].start() if idx + 1 < len(matches) else len(passage)
            article_title = match.group(1).strip()
            article_content = passage[start_idx:end_idx].strip()
            sections.append((article_title, article_content))

        if matches[0].start() > 0:
            preamble = passage[:matches[0].start()].strip()
            if preamble:
                sections.insert(0, ("Lời nói đầu / Quy định chung", preamble))

        for art_title, art_content in sections:
            art_words = art_content.split()
            breadcrumb = f"[{doc_name} > {art_title}] " if doc_name else f"[{art_title}] "
            
            if len(art_words) <= max_words:
                chunks.append({"text": f"{breadcrumb}{art_content}", "article": art_title})
            else:
                sentences = SENTENCE_SPLIT_PATTERN.split(art_content)
                curr_words = []
                for sent in sentences:
                    sent = sent.strip()
                    if not sent:
                        continue
                    s_words = sent.split()
                    if len(curr_words) + len(s_words) > max_words and curr_words:
                        chunk_str = " ".join(curr_words)
                        chunks.append({"text": f"{breadcrumb}{chunk_str}", "article": art_title})
                        curr_words = curr_words[-overlap:] if overlap < len(curr_words) else []
                    curr_words.extend(s_words)
                if curr_words:
                    chunk_str = " ".join(curr_words)
                    chunks.append({"text": f"{breadcrumb}{chunk_str}", "article": art_title})
    else:
        header = f"[{doc_name}] " if doc_name else ""
        start = 0
        while start < len(words):
            end = min(start + max_words, len(words))
            chunk_str = " ".join(words[start:end])
            chunks.append({"text": f"{header}{chunk_str}", "article": doc_name})
            if end == len(words):
                break
            start += (max_words - overlap)

    return chunks if chunks else [{"text": f"{doc_name}. {passage}", "article": doc_name}]

def process_corpus(force_recompute: bool = False) -> List[Dict[str, Any]]:
    if not force_recompute and PROCESSED_CHUNKS_PATH.exists():
        try:
            cached = load_json(PROCESSED_CHUNKS_PATH)
            if cached and len(cached) > 0:
                print(f"[Preprocessing] Nạp {len(cached)} chunks đã xử lý từ cache.")
                return cached
        except Exception:
            pass

    empty_docs = set(corpus_metadata.get("empty_doc_ids", []))
    file_list = glob.glob(str(CORPUS_DIR / "context_*.json"))
    if not file_list:
        file_list = glob.glob(str(CORPUS_DIR / "**" / "context_*.json"), recursive=True)
    if not file_list and IS_KAGGLE:
        file_list = glob.glob("/kaggle/input/**/context_*.json", recursive=True)

    print(f"[Preprocessing] Bắt đầu cắt đoạn theo Context-Aware Chunking trên {len(file_list)} tệp...")
    processed_chunks = []
    for file_path in tqdm(file_list, desc="Chunking Contexts"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                doc_data = json.load(f)
            doc_id = str(doc_data.get('id', ''))
            doc_name = doc_data.get('name', '')
            passage = doc_data.get('passage', '')
            if not doc_id or doc_id in empty_docs:
                continue
            cleaned = normalize_vietnamese_text(passage)
            if not cleaned:
                continue
            chunks = context_aware_chunk_passage(cleaned, doc_name=doc_name, max_words=MAX_CHUNK_WORDS, overlap=CHUNK_OVERLAP)
            for i, c_item in enumerate(chunks):
                processed_chunks.append({
                    "chunk_id": f"{doc_id}_{i}",
                    "doc_id": doc_id,
                    "doc_name": doc_name,
                    "article": c_item["article"],
                    "text": c_item["text"],
                    "segmented_text": word_segment(c_item["text"])
                })
        except Exception:
            continue

    print(f"[Preprocessing] Đã tạo thành công {len(processed_chunks)} context-enriched chunks.")
    if processed_chunks:
        save_json(processed_chunks, PROCESSED_CHUNKS_PATH, indent=2)
    return processed_chunks

corpus_chunks = process_corpus()


In [ ]:
# =====================================================================
# 6. BƯỚC 2: FIRST-STAGE HIGH-RECALL HYBRID RETRIEVAL (BM25 + BGE-M3 + RRF)
# =====================================================================
class HybridRetriever:
    def __init__(self, corpus_chunks: List[Dict[str, Any]], dense_model_name: str = DENSE_MODEL_NAME):
        self.corpus_chunks = corpus_chunks
        self.bm25 = None
        self.dense_model = None
        self.dense_embeddings = None
        self.dense_model_name = dense_model_name
        self._build_bm25_index()
        self._build_dense_index()

    def _build_bm25_index(self):
        if BM25_INDEX_PATH.exists():
            try:
                with open(BM25_INDEX_PATH, 'rb') as f:
                    self.bm25 = pickle.load(f)
                print("[BM25] Nạp chỉ mục BM25 từ cache.")
                return
            except Exception:
                pass
        print(f"[BM25] Đang xây dựng chỉ mục BM25 qua {len(self.corpus_chunks)} chunks...")
        tokenized_corpus = [c['segmented_text'].lower().split() for c in self.corpus_chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump(self.bm25, f)
        print("[BM25] Chỉ mục BM25 hoàn tất.")

    def _build_dense_index(self):
        print(f"🔄 Đang tải Dense Model: {self.dense_model_name}...")
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        
        self.dense_model = SentenceTransformer(
            self.dense_model_name, 
            device=DEVICE,
            model_kwargs={"torch_dtype": torch.float16} if DEVICE == 'cuda' else {}
        )
        # Giới hạn độ dài token để tránh tràn VRAM
        self.dense_model.max_seq_length = 512
        
        texts = [c['text'] for c in self.corpus_chunks]
        print(f"🔄 Đang encode {len(texts)} chunks (batch_size=8)...")
        
        with torch.inference_mode():
            self.dense_embeddings = self.dense_model.encode(
                texts,
                batch_size=8 if DEVICE == 'cuda' else 8,
                show_progress_bar=True,
                normalize_embeddings=True,
                convert_to_numpy=True  # Lưu về CPU RAM thay vì giữ trên VRAM
            )
        
        # Dọn dẹp VRAM sau khi index xong
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        print(f"✅ Đã tạo xong Dense Index với shape: {self.dense_embeddings.shape}")



    def search(self, question: str, top_k: int = FIRST_STAGE_TOP_K) -> List[Dict[str, Any]]:
        # 1. BM25 Search
        q_tokens = word_segment(question).lower().split()
        bm25_scores = self.bm25.get_scores(q_tokens)
        bm25_ranked = np.argsort(bm25_scores)[::-1][:top_k * 2]

        # 2. Dense Search (nếu có)
        dense_ranked = []
        if self.dense_embeddings is not None:
            if self.dense_model is None:
                self.dense_model = SentenceTransformer(self.dense_model_name, device=DEVICE)
            q_emb = self.dense_model.encode([question], normalize_embeddings=True)[0]
            sims = np.dot(self.dense_embeddings, q_emb)
            dense_ranked = np.argsort(sims)[::-1][:top_k * 2]

        # 3. Reciprocal Rank Fusion (RRF)
        rrf_scores = {}
        for rank, idx in enumerate(bm25_ranked):
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (60.0 + rank + 1)
        for rank, idx in enumerate(dense_ranked):
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (60.0 + rank + 1)

        sorted_indices = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        candidates = []
        for idx, score in sorted_indices:
            c = dict(self.corpus_chunks[idx])
            c["rrf_score"] = score
            candidates.append(c)
        return candidates

# Khởi tạo Retriever
retriever = HybridRetriever(corpus_chunks)


In [ ]:
# =====================================================================
# 7. BƯỚC 3: SECOND-STAGE DEEP CROSS-ENCODER RERANKING
# =====================================================================
class DeepReranker:
    def __init__(self, model_name: str = RERANKER_MODEL_NAME):
        self.model_name = model_name
        self.model = None

    def _load_model(self):
        if self.model is None:
            print(f"[Reranker] Đang nạp mô hình Cross-Encoder: {self.model_name}...")
            self.model = CrossEncoder(self.model_name, device=DEVICE, max_length=512)

    def rerank(self, question: str, candidates: List[Dict[str, Any]], top_n: int = RERANK_TOP_K) -> List[Dict[str, Any]]:
        if not candidates:
            return []
        if not HAS_SENTENCE_TRANSFORMERS:
            return candidates[:top_n]
        try:
            self._load_model()
            pairs = [[question, c['text']] for c in candidates]
            scores = self.model.predict(pairs, show_progress_bar=False)
            for i, s in enumerate(scores):
                candidates[i]['rerank_score'] = float(s)
            reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0.0), reverse=True)
            return reranked[:top_n]
        except Exception as e:
            print(f"[Reranker Error] {e}. Fallback to top RRF.")
            return candidates[:top_n]

reranker = DeepReranker()


In [ ]:
# =====================================================================
# 8. BƯỚC 4: STYLE-ALIGNED LEGAL ANSWER GENERATION (QWEN2.5-3B-INSTRUCT)
# =====================================================================
class LegalAnswerGenerator:
    def __init__(self, model_name: str = GENERATOR_MODEL_NAME):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None

    def _load_generator(self):
        if self.model is None:
            print(f"[Generator] Đang nạp mô hình sinh câu trả lời: {self.model_name}...")
            try:
                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    torch_dtype=TORCH_DTYPE,
                    device_map="auto" if DEVICE == "cuda" else None,
                    trust_remote_code=True
                )
            except Exception as e:
                print(f"[Generator] Không thể nạp {self.model_name} ({e}). Thử mô hình dự phòng...")
                self.tokenizer = AutoTokenizer.from_pretrained(FALLBACK_GENERATOR_NAME, trust_remote_code=True)
                self.model = AutoModelForCausalLM.from_pretrained(
                    FALLBACK_GENERATOR_NAME,
                    torch_dtype=TORCH_DTYPE,
                    device_map="auto" if DEVICE == "cuda" else None,
                    trust_remote_code=True
                )

    def generate_answer(self, question: str, top_contexts: List[Dict[str, Any]]) -> str:
        """
        Sinh câu trả lời theo đúng phong cách văn xuôi pháp lý của Thư Viện Pháp Luật:
        - Căn cứ theo Điều... Khoản... [Tên Luật]
        - Trích dẫn nguyên văn điều luật
        - Kết luận ngắn gọn trả lời vào câu hỏi (Theo đó,... / Như vậy,...)
        """
        if not top_contexts:
            return "Căn cứ theo quy định của pháp luật hiện hành, chưa đủ cơ sở để giải đáp nội dung câu hỏi."

        context_text = "\n\n".join([c['text'] for c in top_contexts])

        # Fallback Extractive Template nếu không có GPU hoặc chạy trên CPU
        if not HAS_TRANSFORMERS or DEVICE == 'cpu':
            primary_ctx = top_contexts[0]
            c_text = primary_ctx['text']
            body = re.sub(r'^\[.*?\]\s*', '', c_text)
            doc_ref = primary_ctx.get('doc_name', '')
            art_ref = primary_ctx.get('article', '')
            return f"Căn cứ theo {art_ref} {doc_ref}, quy định cụ thể như sau:\n{body}\nTheo đó, các quy định trên được áp dụng trực tiếp để giải quyết vấn đề câu hỏi đặt ra."

        try:
            self._load_generator()
            system_prompt = (
                "Bạn là chuyên gia tư vấn pháp luật Việt Nam. Dựa vào ngữ cảnh điều luật được cung cấp, "
                "hãy trả lời câu hỏi bằng văn xuôi tự nhiên theo đúng cấu trúc chuẩn:\n"
                "1. Mở đầu bằng căn cứ pháp lý rõ ràng: 'Căn cứ theo Điều... Khoản... [Tên văn bản] quy định như sau:'\n"
                "2. Trích dẫn đầy đủ, chính xác nguyên văn nội dung điều luật liên quan.\n"
                "3. Đoạn kết luận trả lời thẳng vào câu hỏi bắt đầu bằng: 'Theo đó,...' hoặc 'Như vậy,...'\n"
                "LƯU Ý: Giữ nguyên văn các thuật ngữ và câu chữ của điều luật, không thêm các lời chào hỏi lan man."
            )
            user_prompt = f"Ngữ cảnh pháp lý:\n{context_text}\n\nCâu hỏi: {question}\n\nCâu trả lời:"

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
            prompt_text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.tokenizer(prompt_text, return_tensors="pt").to(DEVICE)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=GEN_TEMPERATURE,
                    do_sample=False,  # Greedy decoding tối ưu cho METEOR/ROUGE-L
                    repetition_penalty=1.05
                )
            gen_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            return gen_text
        except Exception as e:
            print(f"[Generation Error] {e}. Sử dụng Extractive Fallback.")
            primary_ctx = top_contexts[0]
            return f"Căn cứ theo {primary_ctx.get('article', '')} {primary_ctx.get('doc_name', '')}:\n{primary_ctx['text']}"

generator = LegalAnswerGenerator()


In [ ]:
# =====================================================================
# 9. BƯỚC 5: ĐÁNH GIÁ OFFLINE TRÊN TẬP VALIDATION (METEOR & ROUGE-L)
# =====================================================================
def evaluate_on_train_split(sample_size: int = 50):
    if not TRAIN_PATH.exists():
        print(f"[Validation] Không tìm thấy tập train tại: {TRAIN_PATH}")
        return

    print(f"[Validation] Đang nạp tập huấn luyện từ: {TRAIN_PATH}...")
    train_data = load_json(TRAIN_PATH)
    all_keys = list(train_data.keys())
    random.seed(RANDOM_SEED)
    val_keys = random.sample(all_keys, min(sample_size, len(all_keys)))
    print(f"[Validation] Đánh giá trên {len(val_keys)} mẫu ngẫu nhiên...")

    ground_truth = {}
    predictions = {}

    for q_id in tqdm(val_keys, desc="Chạy Validation"):
        item = train_data[q_id]
        question = item.get('question', '')
        ref_answer = item.get('answer', '')
        ground_truth[q_id] = {'answer': ref_answer}

        # Pipeline: Retrieve -> Rerank -> Generate
        cands = retriever.search(question, top_k=FIRST_STAGE_TOP_K)
        pinned = reranker.rerank(question, cands, top_n=RERANK_TOP_K)
        pred_ans = generator.generate_answer(question, pinned)
        predictions[q_id] = {'answer': pred_ans}

    scores = evaluate_predictions(ground_truth, predictions)
    print("=" * 60)
    print("             KẾT QUẢ ĐÁNH GIÁ OFFLINE CHUẨN BTC           ")
    print("=" * 60)
    print(f"  - METEOR Score (Độ đo chính) : {scores['meteor']:.4f}")
    print(f"  - ROUGE-L Score (Độ đo phụ)  : {scores['rouge_l']:.4f}")
    print("=" * 60)

# Chạy thử nghiệm đánh giá (mặc định 20 mẫu để kiểm tra nhanh luồng chạy)
evaluate_on_train_split(sample_size=20)


In [ ]:
# =====================================================================
# 10. BƯỚC 6: CHẠY DỰ ĐOÁN TOÀN DIỆN & XUẤT SUBMISSION (PUBLIC-OFFICIAL.JSON)
# =====================================================================
def generate_submission(batch_limit: Optional[int] = None):
    if not TEST_PATH.exists():
        print(f"[Submission] Không tìm thấy file test tại: {TEST_PATH}")
        return

    print(f"[Submission] Đang nạp tệp kiểm tra: {TEST_PATH}...")
    test_data = load_json(TEST_PATH)
    print(f"[Submission] Tổng số câu hỏi cần dự đoán: {len(test_data)}")

    submission = {}
    keys = list(test_data.keys())
    if batch_limit:
        keys = keys[:batch_limit]

    for q_id in tqdm(keys, desc="Generating Submissions"):
        item = test_data[q_id]
        question = item.get('question', '')
        if not question:
            submission[q_id] = {"question": "", "answer": ""}
            continue

        # Pipeline: Hybrid Retrieve -> Deep Rerank -> LLM Generate
        candidates = retriever.search(question, top_k=FIRST_STAGE_TOP_K)
        pinned_articles = reranker.rerank(question, candidates, top_n=RERANK_TOP_K)
        final_answer = generator.generate_answer(question, pinned_articles)

        submission[q_id] = {
            "question": question,
            "answer": final_answer
        }

    # Lưu file submission.json
    save_json(submission, SUBMISSION_PATH, indent=4)
    print(f"[Submission] Đã xuất thành công: {SUBMISSION_PATH} ({len(submission)} mẫu)")

    # Nén tệp submission.zip chuẩn bị nộp bài
    with zipfile.ZipFile(SUBMISSION_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(SUBMISSION_PATH, arcname="submission.json")
    print(f"[Submission] Đã đóng gói tệp nộp bài: {SUBMISSION_ZIP_PATH}")

# Thực thi sinh file nộp bài
generate_submission()
